# APGGS 10-Dataset Ablation Study

This notebook runs the APGGS ablation evaluation with:

- Dataset-specific generation cutoff determined by **Full APGGS**
- Five configurations: Full APGGS, Without PMG, Without Double Bridge, Without Adaptive Control, Basic GA
- Time + convergence measurement
- 10 dataset-specific convergence graphs
- 10 individual comparison tables
- 1 master workbook and CSV outputs
- Time summary and convergence summary
- Overall execution-time vs final-gap figure
- Complete raw JSON experiment record

**Important:** the script expects exactly 10 datasets. Pass them explicitly in the configuration/command section, or place exactly 10 `.tsp/.atsp` files in the data directory.


In [20]:
# ============================================================
# APGGS 10-DATASET ABLATION STUDY
# Time + Convergence + Fixed Full-APGGS Generation Cutoff
#
# Built to preserve the existing APGGS source implementation
# and the existing ablation definitions from:
#   - ADAPTIVE_PROBEGENETICALGORITHM(2).ipynb
#   - APGGS_EVALUATION_ABLATION_STUDY.ipynb
#
# Required packages:
#   python -m pip install numpy pandas matplotlib openpyxl tsplib95 elkai
#
# Usage examples:
#   python apggs_10_dataset_ablation.py
#
#   python apggs_10_dataset_ablation.py --data-dir data \
#       --dataset berlin52 \
#       --dataset a280 \
#       --dataset bays29 \
#       --dataset u574 \
#       --dataset d1655 \
#       --dataset eil51 \
#       --dataset eil76 \
#       --dataset kroa100 \
#       --dataset pr76 \
#       --dataset rat99
#
# If no --dataset arguments are supplied, the script will use the
# .tsp/.atsp files in --data-dir ONLY when exactly 10 are present.
# This prevents silently inventing/replacing the dataset list.

In [21]:
# ============================================================

from __future__ import annotations

import argparse
import ast
import json
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tsplib95
# ============================================================
# GLOBAL EXPERIMENT SETTINGS

In [22]:
# ============================================================

DEFAULT_SOURCE_NOTEBOOKS = [
    Path("ADAPTIVE_PROBEGENETICALGORITHM(2).ipynb"),
    Path("ADAPTIVE_PROBEGENETICALGORITHM.ipynb"),
]

DEFAULT_DATA_DIR = Path("data")
OUTPUT_ROOT = Path("APGGS_ABLATION_10_DATASETS")

SEEDS = [42]

REQUIRED_VARIANTS = [
    "Full APGGS",
    "Without PMG",
    "Without Double Bridge",
    "Without Adaptive Control",
    "Basic GA",
]

FLOAT_TOL = 1e-9
# ============================================================
# EXISTING ABLATION DEFINITIONS — PRESERVED

In [23]:
# ============================================================

@dataclass(frozen=True)
class AblationConfig:
    use_probe: bool = True
    use_pmg: bool = True
    use_double_bridge: bool = True
    use_selective_lkh: bool = True
    use_adaptive_control: bool = True
    basic_ga: bool = False


ABLATION_CONFIGS = {
    "Full APGGS": AblationConfig(),
    "Without PMG": AblationConfig(
        use_pmg=False,
        use_selective_lkh=False,
    ),
    "Without Double Bridge": AblationConfig(
        use_double_bridge=False,
        use_selective_lkh=False,
    ),
    "Without Adaptive Control": AblationConfig(
        use_adaptive_control=False,
        use_selective_lkh=False,
    ),
    "Basic GA": AblationConfig(
        use_probe=False,
        use_pmg=False,
        use_double_bridge=False,
        use_selective_lkh=False,
        use_adaptive_control=False,
        basic_ga=True,
    ),
}
# ============================================================
# SOURCE NOTEBOOK LOADER
# Preserves the evaluation notebook's AST-loading approach,
# while supporting the uploaded "(2)" filename.

In [24]:
# ============================================================

def resolve_source_notebook(explicit_path: Optional[Path] = None) -> Path:
    if explicit_path is not None:
        path = explicit_path.expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(
                f"Source APGGS notebook not found: {path}"
            )
        return path

    for candidate in DEFAULT_SOURCE_NOTEBOOKS:
        path = candidate.expanduser().resolve()
        if path.exists():
            return path

    raise FileNotFoundError(
        "Could not find the APGGS source notebook. Expected one of:\n"
        + "\n".join(f"  - {p}" for p in DEFAULT_SOURCE_NOTEBOOKS)
    )


def load_source_api(path: Path) -> dict[str, Any]:
    notebook = json.loads(path.read_text(encoding="utf-8"))

    source_namespace: dict[str, Any] = {
        "__name__": "__apggs_source__",
        "__file__": str(path),
        "np": np,
        "os": os,
        "random": random,
        "time": time,
        "Path": Path,
        "pd": pd,
        "tsplib95": tsplib95,
    }

    try:
        import elkai

        has_elkai = True
    except Exception:
        elkai = None
        has_elkai = False

    source_namespace["HAS_ELKAI"] = has_elkai
    source_namespace["elkai"] = elkai

    # Keep the source assignments exactly as the evaluation notebook
    # does: configuration assignments are imported, while notebook UI
    # and interactive file-upload cells are skipped.
    allowed_assignments = {"TSPLIB_BEST_KNOWN", "CONFIG"}

    for cell_index, cell in enumerate(notebook.get("cells", [])):
        raw_source = "".join(cell.get("source", []))

        # Remove notebook shell commands, e.g. !pip install ...
        source = "".join(
            line
            for line in raw_source.splitlines(True)
            if not line.lstrip().startswith("!")
        )

        if not source.strip():
            continue

        try:
            tree = ast.parse(
                source,
                filename=f"{path.name}:cell_{cell_index}",
            )
        except SyntaxError as exc:
            raise RuntimeError(
                f"Could not parse source notebook cell {cell_index}: {exc}"
            ) from exc

        selected_nodes = []

        for node in tree.body:

            # Skip Google Colab imports.
            if (
                isinstance(node, ast.ImportFrom)
                and node.module == "google.colab"
            ):
                continue

            if (
                isinstance(node, ast.Import)
                and any(
                    alias.name.startswith("google.colab")
                    for alias in node.names
                )
            ):
                continue

            # Preserve imports/classes/functions.
            if isinstance(
                node,
                (
                    ast.Import,
                    ast.ImportFrom,
                    ast.ClassDef,
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                ),
            ):
                selected_nodes.append(node)
                continue

            # Preserve only the source notebook's main configuration
            # assignments, matching the evaluation notebook behavior.
            if isinstance(node, ast.Assign):
                targets = [
                    target.id
                    for target in node.targets
                    if isinstance(target, ast.Name)
                ]
                if any(name in allowed_assignments for name in targets):
                    selected_nodes.append(node)
                continue

            if isinstance(node, ast.AnnAssign):
                if (
                    isinstance(node.target, ast.Name)
                    and node.target.id in allowed_assignments
                ):
                    selected_nodes.append(node)

        if not selected_nodes:
            continue

        module = ast.Module(
            body=selected_nodes,
            type_ignores=[],
        )
        ast.fix_missing_locations(module)

        exec(
            compile(module, str(path), "exec"),
            source_namespace,
            source_namespace,
        )

    required = [
        "TSPInstance",
        "load_tsplib_instance",
        "GAConfig",
        "build_knn_lists",
        "generate_initial_population",
        "create_individual",
        "run_adaptive_ga",
        "PredictiveMemoryGraph",
        "AdaptiveOperatorSelector",
        "StagnationDetector",
        "nearest_neighbor_tour",
        "two_opt",
        "best_known",
    ]

    missing = [
        name
        for name in required
        if name not in source_namespace
    ]

    if missing:
        raise RuntimeError(
            "Source APGGS API is incomplete. Missing definitions:\n"
            + "\n".join(f"  - {name}" for name in missing)
        )

    return source_namespace
# ============================================================
# DATASET HANDLING

In [25]:
# ============================================================

def find_dataset_file(data_dir: Path, requested: str) -> Path:
    candidate = Path(requested).expanduser()

    if candidate.exists():
        return candidate.resolve()

    requested_lower = requested.lower()
    for extension in (".tsp", ".atsp"):
        direct = data_dir / f"{requested_lower}{extension}"
        if direct.exists():
            return direct.resolve()

    for path in sorted(data_dir.glob("*")):
        if not path.is_file():
            continue
        if path.suffix.lower() not in {".tsp", ".atsp"}:
            continue
        if path.stem.lower() == requested_lower:
            return path.resolve()

    raise FileNotFoundError(
        f"Could not find dataset '{requested}' in {data_dir.resolve()}."
    )


def discover_exactly_ten_datasets(data_dir: Path) -> list[Path]:
    files = sorted(
        [
            path.resolve()
            for path in data_dir.iterdir()
            if path.is_file()
            and path.suffix.lower() in {".tsp", ".atsp"}
        ],
        key=lambda p: p.stem.lower(),
    )

    if len(files) != 10:
        raise RuntimeError(
            f"Automatic dataset discovery requires exactly 10 "
            f".tsp/.atsp files in {data_dir.resolve()}, but found {len(files)}.\n"
            "Pass exactly 10 datasets using --dataset to avoid ambiguity."
        )

    return files


def load_instance(source_api: dict[str, Any], path: Path):
    instance = source_api["load_tsplib_instance"](str(path))
    dataset_name = str(
        instance.name or path.stem
    ).strip().lower()

    return dataset_name, instance
# ============================================================
# SHARED INITIAL POPULATION
# Preserves the existing evaluation method and guarantees the
# same population/seed across all variants for a dataset.

In [26]:
# ============================================================

def build_standard_population(
    source_api: dict[str, Any],
    instance,
    seed: int,
):
    rng = random.Random(seed)

    config = source_api["GAConfig"](
        seed=seed
    )

    neighbors = source_api["build_knn_lists"](
        instance.dist,
        config.k_neighbors,
    )

    seed_tour = source_api["nearest_neighbor_tour"](
        instance.dist,
        start=0,
    )

    optimized_seed, _ = source_api["two_opt"](
        seed_tour,
        instance.dist,
        neighbors,
        max_passes=2,
    )

    population = source_api["generate_initial_population"](
        instance,
        optimized_seed,
        config,
        rng,
    )

    return population, neighbors
# ============================================================
# EXACT EXISTING ABLATION VARIANT RUNNER
# The logic mirrors APGGS_EVALUATION_ABLATION_STUDY.ipynb.

In [27]:
# ============================================================

def run_variant(
    source_api: dict[str, Any],
    instance,
    initial_population,
    neighbors,
    config: AblationConfig,
    seed: int,
    max_generations: int,
):
    cfg = source_api["GAConfig"](
        seed=seed
    )
    cfg.max_generations = int(max_generations)

    if not config.use_selective_lkh:
        cfg.lkh_cadence = (
            int(max_generations) + 1
        )

    original = {
        "AdaptiveOperatorSelector":
            source_api["AdaptiveOperatorSelector"],
        "PredictiveMemoryGraph":
            source_api["PredictiveMemoryGraph"],
        "stagnation_observe":
            source_api["StagnationDetector"].observe,
    }

    try:

        # --------------------------------------------------------
        # WITHOUT ADAPTIVE CONTROL
        # --------------------------------------------------------
        if not config.use_adaptive_control:

            class FixedSelector:

                def __init__(self, rng):
                    self.plays = {
                        "weak_edge_surgery": 0,
                        "predictive_or_opt": 0,
                        "two_opt": 0,
                    }

                def select(self):
                    return "two_opt"

                def update(
                    self,
                    operator,
                    relative_gain,
                ):
                    pass

            source_api[
                "AdaptiveOperatorSelector"
            ] = FixedSelector

        # --------------------------------------------------------
        # WITHOUT DOUBLE BRIDGE
        # --------------------------------------------------------
        if not config.use_double_bridge:

            def never_stagnant(
                self,
                best_cost,
                diversity,
            ):
                return False

            source_api[
                "StagnationDetector"
            ].observe = never_stagnant

        # --------------------------------------------------------
        # WITHOUT PMG / WITHOUT PROBE
        # --------------------------------------------------------
        if (
            not config.use_pmg
            or not config.use_probe
        ):
            base_pmg = original[
                "PredictiveMemoryGraph"
            ]

            use_memory = config.use_pmg
            use_probe = config.use_probe

            class AblatedPredictiveMemoryGraph(
                base_pmg
            ):

                def update(self, population):
                    if use_memory:
                        return super().update(
                            population
                        )

                def edge_quality(self, a, b):
                    return (
                        super().edge_quality(a, b)
                        if use_memory
                        else 0.0
                    )

                def predicted_successors(
                    self,
                    city,
                    k=5,
                ):
                    return (
                        super().predicted_successors(
                            city,
                            k,
                        )
                        if use_memory
                        and use_probe
                        else []
                    )

                def elite_segments(
                    self,
                    top_k=30,
                ):
                    return (
                        super().elite_segments(
                            top_k
                        )
                        if use_memory
                        else []
                    )

            source_api[
                "PredictiveMemoryGraph"
            ] = AblatedPredictiveMemoryGraph

        # --------------------------------------------------------
        # BASIC GA
        # Exact parameter changes from evaluation notebook.
        # --------------------------------------------------------
        if config.basic_ga:
            cfg.mutation_rate = 0.15
            cfg.offspring_local_search_probability = 0.15
            cfg.adaptive_elite_count = 0
            cfg.memory_elite_fraction = 0.0

        start = time.perf_counter()

        result = source_api["run_adaptive_ga"](
            instance,
            initial_population,
            neighbors,
            cfg,
        )

        runtime = time.perf_counter() - start

        result["runtime"] = float(runtime)
        result["variant"] = variant_label_from_config(
            config
        )
        result["seed"] = int(seed)
        result["max_generations"] = int(
            max_generations
        )

        assert result["generations"] <= int(
            max_generations
        )

        return result

    finally:
        source_api[
            "AdaptiveOperatorSelector"
        ] = original[
            "AdaptiveOperatorSelector"
        ]

        source_api[
            "PredictiveMemoryGraph"
        ] = original[
            "PredictiveMemoryGraph"
        ]

        source_api[
            "StagnationDetector"
        ].observe = original[
            "stagnation_observe"
        ]


def variant_label_from_config(
    config: AblationConfig,
) -> str:

    if config.basic_ga:
        return "Basic GA"

    if not config.use_pmg:
        return "Without PMG"

    if not config.use_double_bridge:
        return "Without Double Bridge"

    if not config.use_adaptive_control:
        return "Without Adaptive Control"

    return "Full APGGS"
# ============================================================
# OPTIMUM / CONVERGENCE HELPERS

In [28]:
# ============================================================

def first_optimum_generation(
    history,
    optimum: Optional[float],
) -> Optional[int]:

    if optimum is None:
        return None

    for generation, cost in enumerate(history):
        if float(cost) <= float(optimum) + FLOAT_TOL:
            return int(generation)

    return None


def best_cost_at_generation(
    history,
    generation: int,
) -> float:

    if not history:
        return float("nan")

    idx = min(
        max(int(generation), 0),
        len(history) - 1,
    )

    # History is best-so-far, so history[idx] is
    # the best cost available at the requested generation.
    return float(history[idx])


def gap_percent(
    cost: float,
    optimum: Optional[float],
) -> float:

    if optimum is None:
        return float("nan")

    return (
        (float(cost) - float(optimum))
        / float(optimum)
        * 100.0
    )


def cutoff_status(
    result: dict[str, Any],
    cutoff_generation: int,
    optimum: Optional[float],
) -> str:

    first_opt = first_optimum_generation(
        result["history"],
        optimum,
    )

    if first_opt is not None:
        return "Optimum reached"

    if (
        int(result["generations"])
        < int(cutoff_generation)
    ):
        reason = str(
            result.get(
                "stop_reason",
                "Stopped early",
            )
        )
        return f"Stopped early: {reason}"

    return "Evaluated through cutoff"
# ============================================================
# RESULT RECORD

In [29]:
# ============================================================

def summarize_result(
    dataset_name: str,
    variant: str,
    seed: int,
    result: dict[str, Any],
    evaluation_cutoff: int,
    optimum: Optional[float],
    full_optimum_generation: Optional[int],
    full_reached_optimum: bool,
) -> dict[str, Any]:

    history = list(
        map(float, result.get("history", []))
    )

    first_optimum = first_optimum_generation(
        history,
        optimum,
    )

    final_cost = best_cost_at_generation(
        history,
        evaluation_cutoff,
    )

    actual_generations = int(
        result.get("generations", 0)
    )

    best_generation = int(
        result.get("best_generation", 0)
    )

    runtime = float(
        result.get("runtime", float("nan"))
    )

    reached = (
        first_optimum is not None
        and first_optimum <= evaluation_cutoff
    )

    # Because the source APGGS runner stops immediately
    # when the known optimum is reached, the measured runtime
    # of that run is also its time-to-optimum.
    time_to_optimum = (
        runtime
        if reached
        else float("nan")
    )

    return {
        "Dataset": dataset_name,
        "Variant": variant,
        "Seed": int(seed),
        "Known Optimum": (
            float(optimum)
            if optimum is not None
            else float("nan")
        ),
        "Full APGGS First Optimum Gen": (
            int(full_optimum_generation)
            if full_optimum_generation is not None
            else float("nan")
        ),
        "Full APGGS Reached Optimum": bool(
            full_reached_optimum
        ),
        "Evaluation Cutoff Gen": int(
            evaluation_cutoff
        ),
        "Actual Generations Executed": actual_generations,
        "Final Generation": int(
            min(
                evaluation_cutoff,
                actual_generations,
            )
        ),
        "Best Cost at Cutoff": float(
            final_cost
        ),
        "Best Generation": best_generation,
        "Final Gap (%)": float(
            gap_percent(
                final_cost,
                optimum,
            )
        ),
        "First Optimum Generation": (
            int(first_optimum)
            if reached
            else float("nan")
        ),
        "Time (s)": runtime,
        "Time to Optimum (s)": (
            float(time_to_optimum)
            if reached
            else float("nan")
        ),
        "Reached Optimum?": bool(reached),
        "Stop/Evaluation Status": cutoff_status(
            result,
            evaluation_cutoff,
            optimum,
        ),
        "Stop Reason": str(
            result.get(
                "stop_reason",
                "",
            )
        ),
        "Max Generations Allowed": int(
            result.get(
                "max_generations",
                evaluation_cutoff,
            )
        ),
        "History": history,
        "Final Tour": list(
            result.get(
                "best_tour",
                [],
            )
        ),
        "Generation Records": result.get(
            "generation_records",
            [],
        ),
    }
# ============================================================
# JSON-SAFE CONVERSION

In [30]:
# ============================================================

def to_json_safe(value: Any):
    if isinstance(value, dict):
        return {
            str(k): to_json_safe(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_json_safe(v)
            for v in value
        ]

    if isinstance(value, np.ndarray):
        return [
            to_json_safe(v)
            for v in value.tolist()
        ]

    if isinstance(
        value,
        (
            np.integer,
            np.int64,
            np.int32,
        ),
    ):
        return int(value)

    if isinstance(
        value,
        (
            np.floating,
            np.float64,
            np.float32,
        ),
    ):
        return float(value)

    if isinstance(value, (np.bool_,)):
        return bool(value)

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, float):
        if np.isnan(value) or np.isinf(value):
            return None

    return value
# ============================================================
# PLOT: ONE CONVERGENCE GRAPH PER DATASET

In [31]:
# ============================================================

def make_dataset_convergence_plot(
    dataset_name: str,
    results_for_dataset: list[dict[str, Any]],
    output_path: Path,
):

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    cutoff = int(
        results_for_dataset[0][
            "Evaluation Cutoff Gen"
        ]
    )

    optimum = results_for_dataset[0].get(
        "Known Optimum",
        np.nan,
    )

    plt.figure(
        figsize=(12, 7)
    )

    # Preserve the expected comparison order.
    ordered = []
    by_variant = {
        row["Variant"]: row
        for row in results_for_dataset
    }

    for variant in REQUIRED_VARIANTS:
        if variant in by_variant:
            ordered.append(
                by_variant[variant]
            )

    for row in ordered:
        history = row["History"]

        generations = np.arange(
            len(history)
        )

        # Never show data beyond the cutoff.
        mask = generations <= cutoff

        plt.plot(
            generations[mask],
            np.asarray(history)[mask],
            linewidth=2,
            label=row["Variant"],
        )

    if (
        np.isfinite(float(optimum))
    ):
        plt.axhline(
            y=float(optimum),
            linestyle="--",
            linewidth=1.8,
            label=f"Known Best = {float(optimum):,.4f}",
        )

    plt.axvline(
        x=cutoff,
        linestyle=":",
        linewidth=1.5,
        label=f"Evaluation Cutoff = G{cutoff}",
    )

    plt.xlabel(
        "Generation"
    )

    plt.ylabel(
        "Best Tour Cost"
    )

    plt.title(
        f"APGGS Ablation Convergence — {dataset_name.upper()}"
        f"  |  Evaluation Cutoff = G{cutoff}"
    )

    plt.xlim(
        0,
        max(cutoff, 1),
    )

    plt.grid(
        alpha=0.25
    )

    plt.legend(
        loc="best"
    )

    plt.tight_layout()

    plt.savefig(
        output_path,
        dpi=220,
        bbox_inches="tight",
    )

    plt.close()
# ============================================================
# PLOT: OVERALL TIME VS CONVERGENCE
# Uses final gap (%) so datasets with very different cost scales
# can be compared on one figure.

In [32]:
# ============================================================

def make_overall_time_vs_convergence_plot(
    results_df: pd.DataFrame,
    output_path: Path,
):

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    plt.figure(
        figsize=(12, 7)
    )

    for variant in REQUIRED_VARIANTS:
        subset = results_df[
            results_df["Variant"] == variant
        ].copy()

        if subset.empty:
            continue

        plt.scatter(
            subset["Time (s)"],
            subset["Final Gap (%)"],
            s=65,
            label=variant,
            alpha=0.85,
        )

    plt.xlabel(
        "Execution Time (seconds)"
    )

    plt.ylabel(
        "Final Gap (%)"
    )

    plt.title(
        "Overall Time vs Convergence Quality — 10-Dataset APGGS Ablation Study"
    )

    plt.grid(
        alpha=0.25
    )

    plt.legend(
        loc="best"
    )

    plt.tight_layout()

    plt.savefig(
        output_path,
        dpi=220,
        bbox_inches="tight",
    )

    plt.close()
# ============================================================
# OUTPUT TABLES

In [33]:
# ============================================================

TABLE_COLUMNS = [
    "Variant",
    "Evaluation Cutoff Gen",
    "Final Generation",
    "Actual Generations Executed",
    "Best Cost at Cutoff",
    "Best Generation",
    "Known Optimum",
    "Final Gap (%)",
    "First Optimum Generation",
    "Time (s)",
    "Time to Optimum (s)",
    "Reached Optimum?",
    "Stop/Evaluation Status",
    "Stop Reason",
]


def save_individual_table(
    dataset_name: str,
    dataset_df: pd.DataFrame,
    output_dir: Path,
):

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    table_df = dataset_df[
        TABLE_COLUMNS
    ].copy()

    safe_name = dataset_name.lower()

    csv_path = (
        output_dir
        / f"{safe_name}_comparison.csv"
    )

    xlsx_path = (
        output_dir
        / f"{safe_name}_comparison.xlsx"
    )

    table_df.to_csv(
        csv_path,
        index=False,
    )

    with pd.ExcelWriter(
        xlsx_path,
        engine="openpyxl",
    ) as writer:
        table_df.to_excel(
            writer,
            sheet_name="Comparison",
            index=False,
        )

    return csv_path, xlsx_path
# ============================================================
# MASTER TABLE BUILDERS

In [34]:
# ============================================================

def build_master_long_table(
    results_df: pd.DataFrame,
) -> pd.DataFrame:

    return results_df[
        [
            "Dataset",
            "Evaluation Cutoff Gen",
            "Variant",
            "Best Cost at Cutoff",
            "Best Generation",
            "Final Gap (%)",
            "Time (s)",
            "Time to Optimum (s)",
            "First Optimum Generation",
            "Reached Optimum?",
            "Actual Generations Executed",
            "Known Optimum",
            "Stop/Evaluation Status",
        ]
    ].sort_values(
        ["Dataset", "Variant"]
    ).reset_index(drop=True)


def build_master_wide_table(
    results_df: pd.DataFrame,
) -> pd.DataFrame:

    base = (
        results_df[
            [
                "Dataset",
                "Known Optimum",
                "Evaluation Cutoff Gen",
                "Variant",
                "Best Cost at Cutoff",
                "Final Gap (%)",
                "Time (s)",
                "Time to Optimum (s)",
                "First Optimum Generation",
                "Reached Optimum?",
            ]
        ]
        .copy()
    )

    cost = base.pivot(
        index=[
            "Dataset",
            "Known Optimum",
            "Evaluation Cutoff Gen",
        ],
        columns="Variant",
        values="Best Cost at Cutoff",
    )

    gap = base.pivot(
        index=[
            "Dataset",
            "Known Optimum",
            "Evaluation Cutoff Gen",
        ],
        columns="Variant",
        values="Final Gap (%)",
    )

    time_table = base.pivot(
        index=[
            "Dataset",
            "Known Optimum",
            "Evaluation Cutoff Gen",
        ],
        columns="Variant",
        values="Time (s)",
    )

    optimum_gen = base.pivot(
        index=[
            "Dataset",
            "Known Optimum",
            "Evaluation Cutoff Gen",
        ],
        columns="Variant",
        values="First Optimum Generation",
    )

    reached = base.pivot(
        index=[
            "Dataset",
            "Known Optimum",
            "Evaluation Cutoff Gen",
        ],
        columns="Variant",
        values="Reached Optimum?",
    )

    def flatten(prefix, table):
        table = table.copy()
        table.columns = [
            f"{prefix} — {column}"
            for column in table.columns
        ]
        return table

    wide = pd.concat(
        [
            flatten(
                "Best Cost",
                cost,
            ),
            flatten(
                "Gap (%)",
                gap,
            ),
            flatten(
                "Time (s)",
                time_table,
            ),
            flatten(
                "First Optimum Gen",
                optimum_gen,
            ),
            flatten(
                "Optimum Reached",
                reached,
            ),
        ],
        axis=1,
    ).reset_index()

    return wide


def build_time_summary(
    results_df: pd.DataFrame,
) -> pd.DataFrame:

    pivot = results_df.pivot(
        index="Dataset",
        columns="Variant",
        values="Time (s)",
    ).reset_index()

    cutoff = (
        results_df[
            [
                "Dataset",
                "Full APGGS First Optimum Gen",
                "Evaluation Cutoff Gen",
            ]
        ]
        .drop_duplicates()
    )

    pivot = pivot.merge(
        cutoff,
        on="Dataset",
        how="left",
    )

    ordered_columns = [
        "Dataset",
        "Full APGGS First Optimum Gen",
        "Evaluation Cutoff Gen",
        *[
            variant
            for variant in REQUIRED_VARIANTS
            if variant in pivot.columns
        ],
    ]

    return pivot[
        ordered_columns
    ]


def build_convergence_summary(
    results_df: pd.DataFrame,
) -> pd.DataFrame:

    pivot = results_df.pivot(
        index="Dataset",
        columns="Variant",
        values="Best Cost at Cutoff",
    ).reset_index()

    cutoff = (
        results_df[
            [
                "Dataset",
                "Known Optimum",
                "Evaluation Cutoff Gen",
            ]
        ]
        .drop_duplicates()
    )

    pivot = cutoff.merge(
        pivot,
        on="Dataset",
        how="left",
        suffixes=("", " - duplicate"),
    )

    ordered = [
        "Dataset",
        "Known Optimum",
        "Evaluation Cutoff Gen",
        *[
            variant
            for variant in REQUIRED_VARIANTS
            if variant in pivot.columns
        ],
    ]

    return pivot[
        ordered
    ]
# ============================================================
# COMPLETE EXCEL WORKBOOK

In [35]:
# ============================================================

def save_master_workbook(
    results_df: pd.DataFrame,
    master_wide: pd.DataFrame,
    time_summary: pd.DataFrame,
    convergence_summary: pd.DataFrame,
    output_path: Path,
):

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with pd.ExcelWriter(
        output_path,
        engine="openpyxl",
    ) as writer:

        master_wide.to_excel(
            writer,
            sheet_name="Master Wide",
            index=False,
        )

        build_master_long_table(
            results_df
        ).to_excel(
            writer,
            sheet_name="Master Long",
            index=False,
        )

        time_summary.to_excel(
            writer,
            sheet_name="Time Summary",
            index=False,
        )

        convergence_summary.to_excel(
            writer,
            sheet_name="Convergence Summary",
            index=False,
        )

        for dataset in sorted(
            results_df["Dataset"].unique()
        ):
            sheet_name = (
                dataset.upper()
                .replace(" ", "_")
            )[:31]

            table = results_df[
                results_df["Dataset"] == dataset
            ][TABLE_COLUMNS].copy()

            table.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False,
            )
# ============================================================
# VALIDATION

In [36]:
# ============================================================

def validate_results(
    results_df: pd.DataFrame,
    expected_datasets: list[str],
):

    datasets = sorted(
        results_df["Dataset"].unique()
    )

    if len(datasets) != 10:
        raise AssertionError(
            f"Expected exactly 10 datasets, found {len(datasets)}."
        )

    if sorted(expected_datasets) != datasets:
        raise AssertionError(
            "Processed dataset list does not match requested dataset list."
        )

    for dataset in datasets:
        subset = results_df[
            results_df["Dataset"] == dataset
        ]

        variants = sorted(
            subset["Variant"].unique()
        )

        if variants != sorted(
            REQUIRED_VARIANTS
        ):
            raise AssertionError(
                f"{dataset}: required variants mismatch. "
                f"Found {variants}"
            )

        cutoff_values = (
            subset[
                "Evaluation Cutoff Gen"
            ]
            .astype(int)
            .unique()
        )

        if len(cutoff_values) != 1:
            raise AssertionError(
                f"{dataset}: multiple evaluation cutoffs found."
            )

        cutoff = int(
            cutoff_values[0]
        )

        for _, row in subset.iterrows():

            actual = int(
                row["Actual Generations Executed"]
            )

            if actual > cutoff:
                raise AssertionError(
                    f"{dataset}/{row['Variant']} executed "
                    f"{actual} generations > cutoff {cutoff}."
                )

            history = row["History"]

            if len(history) > cutoff + 1:
                raise AssertionError(
                    f"{dataset}/{row['Variant']} contains "
                    f"history beyond cutoff."
                )

    # Exactly 50 rows: 10 datasets x 5 variants.
    if len(results_df) != 50:
        raise AssertionError(
            f"Expected 50 experiment rows, found {len(results_df)}."
        )

    # Full APGGS cutoff rule.
    for dataset in datasets:

        full_row = results_df[
            (results_df["Dataset"] == dataset)
            & (results_df["Variant"] == "Full APGGS")
        ]

        if len(full_row) != 1:
            raise AssertionError(
                f"{dataset}: Full APGGS row missing or duplicated."
            )

        full_row = full_row.iloc[0]

        first_opt = full_row[
            "Full APGGS First Optimum Gen"
        ]

        cutoff = int(
            full_row["Evaluation Cutoff Gen"]
        )

        if pd.notna(first_opt):
            if cutoff != int(first_opt):
                raise AssertionError(
                    f"{dataset}: cutoff {cutoff} does not equal "
                    f"Full APGGS first optimum generation {int(first_opt)}."
                )
        else:
            actual = int(
                full_row["Actual Generations Executed"]
            )
            if cutoff != actual:
                raise AssertionError(
                    f"{dataset}: fallback cutoff {cutoff} does not "
                    f"match Full APGGS termination generation {actual}."
                )

    print(
        "[VALIDATION] All experiment integrity checks passed."
    )
# ============================================================
# DATASET-LEVEL EXPERIMENT

In [37]:
# ============================================================

def run_dataset_experiment(
    source_api: dict[str, Any],
    dataset_path: Path,
    output_dirs: dict[str, Path],
):

    dataset_name, instance = load_instance(
        source_api,
        dataset_path,
    )

    optimum = source_api["best_known"](
        dataset_name
    )

    print("\n" + "=" * 90)
    print(
        f"DATASET: {dataset_name.upper()}"
    )
    print(
        f"File: {dataset_path}"
    )
    print(
        f"Cities: {instance.n}"
    )
    print(
        f"Known Best: "
        f"{optimum if optimum is not None else 'Unknown'}"
    )
    print("=" * 90)

    dataset_records = []

    # --------------------------------------------------------
    # For each seed, Full APGGS first determines the cutoff.
    # --------------------------------------------------------
    for seed in SEEDS:

        (
            shared_population,
            neighbors,
        ) = build_standard_population(
            source_api,
            instance,
            seed,
        )

        full_cfg = ABLATION_CONFIGS[
            "Full APGGS"
        ]

        # First run: Full APGGS gets the normal source
        # maximum of 300 generations. Its first optimum
        # generation determines the dataset-specific cutoff.
        full_result = run_variant(
            source_api,
            instance,
            shared_population,
            neighbors,
            full_cfg,
            seed,
            300,
        )

        full_history = list(
            map(
                float,
                full_result["history"],
            )
        )

        full_first_optimum = first_optimum_generation(
            full_history,
            optimum,
        )

        if full_first_optimum is not None:
            evaluation_cutoff = int(
                full_first_optimum
            )
            full_reached = True
        else:
            evaluation_cutoff = int(
                full_result["generations"]
            )
            full_reached = False

        print(
            f"[FULL APGGS] First optimum generation: "
            f"{full_first_optimum if full_first_optimum is not None else 'NOT REACHED'}"
        )

        print(
            f"[FULL APGGS] Dataset evaluation cutoff: "
            f"G{evaluation_cutoff}"
        )

        # ----------------------------------------------------
        # Re-run Full APGGS at its fixed cutoff only.
        #
        # This ensures the final reported Full APGGS runtime
        # and history correspond to the SAME fixed-budget
        # evaluation used for all other variants.
        #
        # If the original run stopped earlier at the optimum,
        # this second run reproduces the same stopping point.
        # ----------------------------------------------------
        full_eval_result = run_variant(
            source_api,
            instance,
            shared_population,
            neighbors,
            full_cfg,
            seed,
            evaluation_cutoff,
        )

        full_record = summarize_result(
            dataset_name=dataset_name,
            variant="Full APGGS",
            seed=seed,
            result=full_eval_result,
            evaluation_cutoff=evaluation_cutoff,
            optimum=optimum,
            full_optimum_generation=full_first_optimum,
            full_reached_optimum=full_reached,
        )

        dataset_records.append(
            full_record
        )

        # ----------------------------------------------------
        # Ablation variants
        # Each gets the same population, same seed, and ONLY
        # the Full-APGGS-derived generation budget.
        # ----------------------------------------------------
        for variant_label in [
            "Without PMG",
            "Without Double Bridge",
            "Without Adaptive Control",
            "Basic GA",
        ]:

            variant_cfg = ABLATION_CONFIGS[
                variant_label
            ]

            result = run_variant(
                source_api,
                instance,
                shared_population,
                neighbors,
                variant_cfg,
                seed,
                evaluation_cutoff,
            )

            assert result["generations"] <= (
                evaluation_cutoff
            )

            record = summarize_result(
                dataset_name=dataset_name,
                variant=variant_label,
                seed=seed,
                result=result,
                evaluation_cutoff=evaluation_cutoff,
                optimum=optimum,
                full_optimum_generation=full_first_optimum,
                full_reached_optimum=full_reached,
            )

            dataset_records.append(
                record
            )

    # --------------------------------------------------------
    # One seed is the existing evaluation default. If more
    # seeds are later configured, aggregate the seed-level
    # records by keeping the per-run records intact.
    # --------------------------------------------------------
    return dataset_records
# ============================================================
# MAIN

In [38]:
# ============================================================

def parse_args():
    parser = argparse.ArgumentParser(
        description=(
            "Run the APGGS 10-dataset ablation study with "
            "dataset-specific Full-APGGS cutoffs."
        )
    )

    parser.add_argument(
        "--source-notebook",
        type=Path,
        default=None,
        help=(
            "Path to the APGGS source notebook. "
            "Defaults to ADAPTIVE_PROBEGENETICALGORITHM(2).ipynb "
            "or ADAPTIVE_PROBEGENETICALGORITHM.ipynb."
        ),
    )

    parser.add_argument(
        "--data-dir",
        type=Path,
        default=DEFAULT_DATA_DIR,
        help="Directory containing .tsp/.atsp datasets.",
    )

    parser.add_argument(
        "--dataset",
        action="append",
        default=None,
        help=(
            "Dataset stem, dataset filename, or dataset path. "
            "Pass exactly 10 times to explicitly select the 10 datasets."
        ),
    )

    parser.add_argument(
        "--output-dir",
        type=Path,
        default=OUTPUT_ROOT,
        help="Output directory for all tables, graphs and raw results.",
    )

    return parser.parse_args()


def main():
    args = parse_args()

    source_notebook = resolve_source_notebook(
        args.source_notebook
    )

    source_api = load_source_api(
        source_notebook
    )

    print(
        f"Loaded APGGS source definitions from: "
        f"{source_notebook}"
    )

    # --------------------------------------------------------
    # Resolve exactly 10 datasets without silently replacing
    # the user's dataset set.
    # --------------------------------------------------------
    if args.dataset:

        if len(args.dataset) != 10:
            raise ValueError(
                f"--dataset was supplied {len(args.dataset)} times. "
                "Exactly 10 datasets are required."
            )

        dataset_paths = [
            find_dataset_file(
                args.data_dir,
                requested,
            )
            for requested in args.dataset
        ]

    else:
        dataset_paths = (
            discover_exactly_ten_datasets(
                args.data_dir
            )
        )

    dataset_names = []

    for path in dataset_paths:
        name, _ = load_instance(
            source_api,
            path,
        )
        dataset_names.append(name)

    if len(set(dataset_names)) != 10:
        raise ValueError(
            "Dataset names must be unique. "
            f"Resolved names: {dataset_names}"
        )

    # --------------------------------------------------------
    # Output directory structure
    # --------------------------------------------------------
    output_root = (
        args.output_dir
        .expanduser()
        .resolve()
    )

    output_dirs = {
        "root": output_root,
        "graphs": (
            output_root
            / "convergence_graphs"
        ),
        "individual_tables": (
            output_root
            / "individual_tables"
        ),
        "master": (
            output_root
            / "master_results"
        ),
        "raw": (
            output_root
            / "raw_results"
        ),
    }

    for path in output_dirs.values():
        path.mkdir(
            parents=True,
            exist_ok=True,
        )

    # --------------------------------------------------------
    # Batch execution
    # --------------------------------------------------------
    all_records = []

    for index, dataset_path in enumerate(
        dataset_paths,
        start=1,
    ):

        print("\n")
        print(
            f"################ DATASET "
            f"{index}/10 ################"
        )

        records = run_dataset_experiment(
            source_api,
            dataset_path,
            output_dirs,
        )

        all_records.extend(
            records
        )

    results_df = pd.DataFrame(
        all_records
    )

    # --------------------------------------------------------
    # With one seed, this is exactly 50 rows:
    # 10 datasets x 5 variants.
    #
    # With multiple seeds, the detailed raw results are still
    # retained; the requested master comparison is generated
    # from the configured seed-level records.
    # --------------------------------------------------------
    validate_results(
        results_df,
        dataset_names,
    )

    # --------------------------------------------------------
    # Save one individual table + one graph per dataset.
    # --------------------------------------------------------
    individual_paths = []

    for dataset_name in sorted(
        results_df["Dataset"].unique()
    ):

        dataset_df = results_df[
            results_df["Dataset"] == dataset_name
        ].copy()

        csv_path, xlsx_path = save_individual_table(
            dataset_name,
            dataset_df,
            output_dirs["individual_tables"],
        )

        individual_paths.extend(
            [csv_path, xlsx_path]
        )

        make_dataset_convergence_plot(
            dataset_name=dataset_name,
            results_for_dataset=(
                dataset_df.to_dict(
                    orient="records"
                )
            ),
            output_path=(
                output_dirs["graphs"]
                / f"{dataset_name}_convergence.png"
            ),
        )

    # --------------------------------------------------------
    # Master tables
    # --------------------------------------------------------
    master_long = build_master_long_table(
        results_df
    )

    master_wide = build_master_wide_table(
        results_df
    )

    time_summary = build_time_summary(
        results_df
    )

    convergence_summary = (
        build_convergence_summary(
            results_df
        )
    )

    master_long.to_csv(
        output_dirs["master"]
        / "APGGS_10_DATASET_MASTER_TABLE_LONG.csv",
        index=False,
    )

    master_wide.to_csv(
        output_dirs["master"]
        / "APGGS_10_DATASET_MASTER_TABLE.csv",
        index=False,
    )

    time_summary.to_csv(
        output_dirs["master"]
        / "APGGS_10_DATASET_TIME_SUMMARY.csv",
        index=False,
    )

    convergence_summary.to_csv(
        output_dirs["master"]
        / "APGGS_10_DATASET_CONVERGENCE_SUMMARY.csv",
        index=False,
    )

    master_xlsx = (
        output_dirs["master"]
        / "APGGS_10_DATASET_MASTER_TABLE.xlsx"
    )

    save_master_workbook(
        results_df=results_df,
        master_wide=master_wide,
        time_summary=time_summary,
        convergence_summary=convergence_summary,
        output_path=master_xlsx,
    )

    # --------------------------------------------------------
    # Overall time vs convergence figure
    # --------------------------------------------------------
    overall_plot = (
        output_dirs["master"]
        / "overall_time_vs_convergence.png"
    )

    make_overall_time_vs_convergence_plot(
        results_df,
        overall_plot,
    )

    # --------------------------------------------------------
    # Complete machine-readable raw results
    # --------------------------------------------------------
    raw_payload = {
        "source_notebook": str(
            source_notebook
        ),
        "datasets": [
            {
                "dataset": str(path),
                "resolved_name": str(
                    name
                ),
            }
            for path, name in zip(
                dataset_paths,
                dataset_names,
            )
        ],
        "seeds": SEEDS,
        "variants": REQUIRED_VARIANTS,
        "float_tolerance": FLOAT_TOL,
        "results": all_records,
    }

    raw_json = (
        output_dirs["raw"]
        / "complete_experiment_results.json"
    )

    raw_json.write_text(
        json.dumps(
            to_json_safe(
                raw_payload
            ),
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    # --------------------------------------------------------
    # Machine-readable flat run table excluding nested fields
    # --------------------------------------------------------
    flat_export = results_df.drop(
        columns=[
            "History",
            "Final Tour",
            "Generation Records",
        ],
        errors="ignore",
    )

    flat_export.to_csv(
        output_dirs["master"]
        / "APGGS_10_DATASET_ALL_50_RUNS.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Final console summary
    # --------------------------------------------------------
    print("\n" + "=" * 100)
    print(
        "APGGS 10-DATASET ABLATION STUDY COMPLETE"
    )
    print("=" * 100)

    cutoff_summary = (
        results_df[
            results_df["Variant"] == "Full APGGS"
        ][
            [
                "Dataset",
                "Full APGGS First Optimum Gen",
                "Evaluation Cutoff Gen",
                "Reached Optimum?",
                "Time (s)",
            ]
        ]
        .sort_values("Dataset")
    )

    print(
        cutoff_summary.to_string(
            index=False
        )
    )

    print("\n" + "-" * 100)

    print(
        f"Datasets processed       : "
        f"{results_df['Dataset'].nunique()}"
    )

    print(
        f"Variants per dataset     : "
        f"{len(REQUIRED_VARIANTS)}"
    )

    print(
        f"Experiment rows          : "
        f"{len(results_df)}"
    )

    print(
        "Convergence graphs       : 10"
    )

    print(
        "Individual tables        : 10"
    )

    print(
        "Merged master table      : 1"
    )

    print(
        "Time summary table       : 1"
    )

    print(
        "Convergence summary      : 1"
    )

    print(
        "Overall time-convergence : 1"
    )

    print("\nSaved output root:")
    print(
        f"  {output_root}"
    )

    print("\nKey files:")
    print(
        f"  Master XLSX : "
        f"{master_xlsx}"
    )
    print(
        f"  Master CSV  : "
        f"{output_dirs['master'] / 'APGGS_10_DATASET_MASTER_TABLE.csv'}"
    )
    print(
        f"  Time Summary: "
        f"{output_dirs['master'] / 'APGGS_10_DATASET_TIME_SUMMARY.csv'}"
    )
    print(
        f"  Conv Summary: "
        f"{output_dirs['master'] / 'APGGS_10_DATASET_CONVERGENCE_SUMMARY.csv'}"
    )
    print(
        f"  Overall Plot: "
        f"{overall_plot}"
    )
    print(
        f"  Raw JSON    : "
        f"{raw_json}"
    )

    print("\n" + "=" * 100)
    print(
        "FINAL EXPERIMENTAL RULE ENFORCED:"
    )
    print(
        "Full APGGS determines the dataset-specific cutoff; "
        "all five configurations are compared only within that cutoff."
    )
    print("=" * 100)


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--source-notebook SOURCE_NOTEBOOK]
                             [--data-dir DATA_DIR] [--dataset DATASET]
                             [--output-dir OUTPUT_DIR]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\VICTUS\AppData\Roaming\jupyter\runtime\kernel-v391dd0ef2ab904d964c7e7ef9e9df5a6b0eebd070.json


SystemExit: 2

## Running the experiment

Set `--data-dir` and supply **exactly 10** dataset names/paths through the argument parser when running from a terminal.

From a notebook, the simplest option is to edit `sys.argv` in a cell and call `main()`, for example:

```python
import sys
sys.argv = [
    "apggs_10_dataset_ablation.py",
    "--data-dir", "data",
    "--dataset", "berlin52",
    "--dataset", "a280",
    "--dataset", "bays29",
    "--dataset", "u574",
    "--dataset", "d1655",
    "--dataset", "eil51",
    "--dataset", "eil76",
    "--dataset", "kroa100",
    "--dataset", "pr76",
    "--dataset", "rat99",
]
main()
```

Replace the last five names with the five datasets configured in your existing evaluation study if those differ.


## ▶️ Execution Cell — Run This to Start the Study

Run **this cell** to launch the full 10-dataset ablation study.

It sets the dataset list and calls `main()`. All 10 convergence graphs, 10 tables, and 1 master workbook will be saved to `APGGS_ABLATION_10_DATASETS/`.

In [ ]:
import sys

# ── Configure the 10 TSPLIB datasets ────────────────────────────────────
# These are the 10 files extracted from tsplib_instances.zip into data/
sys.argv = [
    "apggs_10_dataset_ablation",
    "--source-notebook", "ADAPTIVE_PROBEGENETICALGORITHM.ipynb",
    "--data-dir", "data",
    "--dataset", "a280",
    "--dataset", "bays29",
    "--dataset", "berlin52",
    "--dataset", "bier127",
    "--dataset", "d657",
    "--dataset", "lin318",
    "--dataset", "pr1002",
    "--dataset", "rat783",
    "--dataset", "rd400",
    "--dataset", "u574",
]

# ── Launch the study ─────────────────────────────────────────────────────
main()


## 📋 Verify Outputs

Run the cell below after the study completes to verify all outputs were generated.

In [ ]:
from pathlib import Path
import pandas as pd

output_root = Path("APGGS_ABLATION_10_DATASETS")

checks = {
    "Convergence graphs (10 PNG)": list((output_root / "convergence_graphs").glob("*.png")),
    "Individual CSV tables (10)": list((output_root / "individual_tables").glob("*.csv")),
    "Individual XLSX tables (10)": list((output_root / "individual_tables").glob("*.xlsx")),
    "Master CSV files": list((output_root / "master_results").glob("*.csv")),
    "Master XLSX workbook": list((output_root / "master_results").glob("*.xlsx")),
    "Raw JSON results": list((output_root / "raw_results").glob("*.json")),
}

print("=" * 70)
print("OUTPUT VERIFICATION")
print("=" * 70)
all_ok = True
for label, files in checks.items():
    ok = bool(files)
    if not ok:
        all_ok = False
    status = "[OK]    " if ok else "[MISSING]"
    print(f"{status}  {label}: {len(files)} file(s)")
    for f in sorted(files):
        print(f"          {f.name}")

print()
if all_ok:
    print("All outputs generated successfully!")
else:
    print("WARNING: Some outputs are missing.")

# Show master table
master_csv = output_root / "master_results" / "APGGS_10_DATASET_MASTER_TABLE.csv"
if master_csv.exists():
    print("\n" + "=" * 70)
    print("MASTER COMPARISON TABLE")
    print("=" * 70)
    df = pd.read_csv(master_csv)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 250)
    print(df.to_string(index=False))


## 📊 Convergence Graphs

Run the cell below to display all 10 convergence graphs inline.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

graphs_dir = Path("APGGS_ABLATION_10_DATASETS") / "convergence_graphs"
graphs = sorted(graphs_dir.glob("*.png"))

print(f"Found {len(graphs)} convergence graphs:\n")
for g in graphs:
    print(f"--- {g.stem} ---")
    display(Image(str(g), width=900))
    print()
